# Building an Artificial Neural Network with Backpropagation

In this notebook, we'll implement a simple Artificial Neural Network (ANN) from scratch using the backpropagation algorithm and test it on a suitable dataset.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

# Set random seed for reproducibility
np.random.seed(42)

## Part 1: Implementing the Neural Network Class

First, let's implement our neural network class with the backpropagation algorithm.

In [ ]:
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.01):
        # Initialize network parameters
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.learning_rate = learning_rate
        
        # Initialize weights and biases
        # Weights between input and hidden layer
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        
        # Weights between hidden and output layer
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))
        
        # Initialize error history
        self.error_history = []
    
    def sigmoid(self, x):
        """Sigmoid activation function"""
        return 1 / (1 + np.exp(-x))
    
    def sigmoid_derivative(self, x):
        """Derivative of sigmoid function"""
        return x * (1 - x)
    
    def forward(self, X):
        """Forward propagation"""
        # Input to hidden layer
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.sigmoid(self.z1)
        
        # Hidden to output layer
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self.sigmoid(self.z2)
        
        return self.a2
    
    def backward(self, X, y, output):
        """Backward propagation (backpropagation)"""
        # Calculate error
        self.error = y - output
        
        # Calculate gradients for output layer
        d_output = self.error * self.sigmoid_derivative(output)
        
        # Calculate gradients for hidden layer
        d_hidden = np.dot(d_output, self.W2.T) * self.sigmoid_derivative(self.a1)
        
        # Update weights and biases
        # Output layer
        self.W2 += self.learning_rate * np.dot(self.a1.T, d_output)
        self.b2 += self.learning_rate * np.sum(d_output, axis=0, keepdims=True)
        
        # Hidden layer
        self.W1 += self.learning_rate * np.dot(X.T, d_hidden)
        self.b1 += self.learning_rate * np.sum(d_hidden, axis=0, keepdims=True)
    
    def train(self, X, y, epochs=10000):
        """Train the neural network"""
        for epoch in range(epochs):
            # Forward propagation
            output = self.forward(X)
            
            # Calculate error
            error = np.mean(np.square(y - output))
            self.error_history.append(error)
            
            # Backward propagation
            self.backward(X, y, output)
            
            # Print progress
            if epoch % 1000 == 0:
                print(f"Epoch {epoch}, Error: {error:.6f}")
    
    def predict(self, X):
        """Make predictions"""
        return self.forward(X)
    
    def plot_error_history(self):
        """Plot the error history during training"""
        plt.figure(figsize=(10, 6))
        plt.plot(self.error_history, color='blue')
        plt.title('Error History during Training', fontsize=14)
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel('Mean Squared Error', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.show()

## Part 2: Preparing the Dataset

We'll use the Iris dataset to test our neural network.

In [ ]:
# Load and prepare the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# One-hot encode the target
encoder = OneHotEncoder(sparse_output=False)
y_one_hot = encoder.fit_transform(y.reshape(-1, 1))

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_one_hot, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Display data shape
print(f"X_train shape: {X_train_scaled.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test_scaled.shape}")
print(f"y_test shape: {y_test.shape}")

## Part 3: Training the Neural Network

Now, let's train our neural network on the Iris dataset.

In [ ]:
# Initialize and train the neural network
input_size = X_train_scaled.shape[1]  # 4 features
hidden_size = 8  # Number of neurons in hidden layer
output_size = y_train.shape[1]  # 3 classes

# Create and train the neural network
nn = NeuralNetwork(input_size, hidden_size, output_size, learning_rate=0.1)
nn.train(X_train_scaled, y_train, epochs=10000)

In [ ]:
# Plot the error history
nn.plot_error_history()

## Part 4: Evaluating the Neural Network

Let's evaluate our network's performance on the test data.

In [ ]:
# Make predictions
predictions = nn.predict(X_test_scaled)

# Convert probabilities to class labels
y_pred = np.argmax(predictions, axis=1)
y_true = np.argmax(y_test, axis=1)

# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

# Generate confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(conf_matrix)

In [ ]:
# Visualize the confusion matrix
plt.figure(figsize=(8, 6))
plt.imshow(conf_matrix, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix', fontsize=14)
plt.colorbar()

# Add labels and ticks
classes = iris.target_names
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes)
plt.yticks(tick_marks, classes)

# Add text annotations
thresh = conf_matrix.max() / 2
for i in range(conf_matrix.shape[0]):
    for j in range(conf_matrix.shape[1]):
        plt.text(j, i, format(conf_matrix[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if conf_matrix[i, j] > thresh else "black")

plt.ylabel('True label', fontsize=12)
plt.xlabel('Predicted label', fontsize=12)
plt.tight_layout()
plt.show()

## Conclusion

In this notebook, we've built an artificial neural network from scratch implementing the backpropagation algorithm. We've covered:

1. The implementation of a neural network class with forward and backward propagation
2. Training the network on the Iris dataset
3. Evaluating the network's performance
4. Visualizing the error history during training

This demonstrates how neural networks learn through the backpropagation algorithm by adjusting weights based on prediction errors. The same principles apply to more complex neural networks used in modern deep learning, though those typically have many more layers and neurons.